# 07 · 多代理協作：Planner + Executor

複雜任務一個 agent 容易迷路。人類會**先列計畫、再一步步做**。我們把它搬給 agent：

- **Planner**：把任務拆成有序子步驟（只規劃、不執行）。
- **Executor**：前幾課的 ReAct 工具 agent，一次專心做一步。
- **Orchestrator**：串流程、累積結果、彙整答案。

## 1. 載入模型 + 執行器（前四課成果濃縮）

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

In [ ]:
from datetime import datetime, timezone, timedelta


def get_current_time() -> str:
    return datetime.now(timezone(timedelta(hours=8))).strftime("%Y-%m-%d %H:%M:%S")


def calculator(expression: str) -> str:
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression or ""):
        return "錯誤：只接受數字與 + - * / ( ) . %"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"錯誤：{e}"


TOOLS = {"get_current_time": get_current_time, "calculator": calculator}
REACT_SYSTEM = """你是會用工具、一步步推理的助手。工具：
- get_current_time()：現在台北時間，無參數。
- calculator(expression)：算式，例 {"expression": "15-9"}。
格式（一次一步）：
Thought: 推理
Action: 工具名
Action Input: JSON
我會回 Observation。足夠時改用：
Thought: 推理
Final Answer: 答案"""


def parse_action(text):
    m = re.search(r"Action:\s*([A-Za-z_]\w*)", text)
    if not m:
        return None, None
    args, ma = {}, re.search(r"Action Input:\s*(\{.*\})", text, re.DOTALL)
    if ma:
        try:
            args = json.loads(ma.group(1))
        except Exception:
            args = {}
    return m.group(1), args


def run_react(question, max_steps=5, verbose=False):
    pad = ""
    for _ in range(max_steps):
        msg = [{"role": "system", "content": REACT_SYSTEM},
               {"role": "user", "content": f"問題：{question}\n\n{pad}"}]
        r = chat(msg, max_new_tokens=256).split("Observation")[0].strip()
        if verbose:
            print(r)
        if "Final Answer:" in r:
            return r.split("Final Answer:")[-1].strip()
        name, args = parse_action(r)
        obs = "（沒解析到 Action）" if not name else (
            TOOLS[name](**args) if name in TOOLS else f"沒有 {name} 這支工具")
        pad += f"{r}\nObservation: {obs}\n"
    return "（達到最大步數）"

## 2. Planner：把任務拆成有序步驟

In [ ]:
def plan(task):
    p = chat([{"role": "user",
               "content": f"把任務拆成 2-4 個有序步驟，每行一步、用「1. 」開頭，"
                          f"不要解釋：\n{task}"}], max_new_tokens=200)
    return re.findall(r"\d+\.\s*(.+)", p)


print(plan("查現在幾點，並算出再過 90 分鐘是幾點幾分"))

## 3. Orchestrator：逐步交給 executor，再彙整

In [ ]:
def orchestrate(task, verbose=True):
    steps = plan(task)
    if verbose:
        print("📋 計畫：")
        for i, s in enumerate(steps, 1):
            print(f"  {i}. {s}")
    results = []
    for i, s in enumerate(steps, 1):
        r = run_react(s)
        results.append(f"步驟{i}（{s}）→ {r}")
        if verbose:
            print(f"\n✅ 步驟 {i} 結果：{r}")
    final = chat([{"role": "user",
                   "content": "根據各步驟結果，彙整成給使用者的最終答案：\n"
                              + "\n".join(results)}])
    return final


print("\n🎯 最終答案：", orchestrate("查現在幾點，並算出『分鐘』數字乘以 60 是多少"))

## 4. Sidebar：Reflection（自我批評再修正）

多代理的近親：同一個 agent **先答、再自我批評、再修正**。一寫一改、自我迭代。

In [ ]:
Q = "用一句話說明為什麼天空是藍色的。"
draft = chat([{"role": "user", "content": Q}])
critique = chat([{"role": "user",
                  "content": f"指出這個回答可以更好的地方（簡短）：\n{draft}"}])
revised = chat([{"role": "user",
                 "content": f"問題：{Q}\n初版：{draft}\n批評：{critique}\n"
                            f"請根據批評給出更好的一句話回答。"}])
print("初版：", draft, "\n批評：", critique, "\n改進：", revised)

## 小結

- **Planner + Executor + Orchestrator**：一個負責拆、一個負責做、一個負責合。
- 分工**降低每個 agent 的認知負擔**，解單一 agent 扛不動的複雜任務。
- **Reflection** 是近親：自答、自評、自改。

下一課：把全部零件組成一個**能用的研究助理**，並接軌真實世界 API。